[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/15_calidad_regresion/15_proyecto_calidad_regresion.ipynb)

# Proyecto 15 · Detectar una regresión de prompt usando solo telemetría

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (todo en memoria). Opcional: `core` + `metrics` para ver tu panel (paso final) |
| **RAM aproximada** | < 500 MB |
| **Prerrequisitos** | Lección `15_calidad_regresion`, proyecto 14 (tu evaluador) y módulos 07 y 10 |
| **Conexión** | Usas el evaluador del 14 como detector. El 16 convertirá esta señal en un SLO de calidad con alertas |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. El jueves se desplegó «prompt v2». Soporte humano dice que «algo raro pasa con los números de seguimiento», pero **no tienes acceso al repositorio de prompts**
(es de otro equipo) y los paneles de latencia y errores están verdes. Solo tienes **telemetría**: spans del agente (con `gen_ai.agent.version`) y eventos de evaluación. Tu misión:

1. **Demostrar con estadística** si hubo regresión y en qué versión.
2. **Localizar las trazas culpables** y describir la causa sin leer el prompt.
3. **Montar la red de seguridad**: dataset de oro desde producción, compuerta de release, vigilancia del comportamiento y un canario con decisión secuencial.

## Datos

Sintéticos y deterministas (semilla 42, `mock`). El **Paso 0** genera (a) 800 tickets de «producción» con la versión v1 (evaluados por el juez, con feedback sintético) para construir el dataset de oro y (b) un **A/B consecutivo** de 500 tickets con v1 y 500 con v2
exportados como spans y eventos (`SPANS`, `EVENTS`) — **de ahí sale la detección «solo telemetría»**. Cadena de datos: stack local (opcional) → memoria.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `build_golden` | Mismos casos y razones que la referencia, sin duplicados ni PII, pesos ≥ 1 | exacto |
| R2 | `compare_versions` | Prueba apareada McNemar: v2 se bloquea, v1 vs v1 pasa, listas distintas lanzan `ValueError` | 4/4 |
| R3 | `fingerprint` y `drift_flags` | Huella exacta y banderas correctas (v2 marcada, v2.1 no marcada en `no_tool_rate`) | exacto / 3/3 |
| R4 | `find_regression` | Versión culpable, tasas de groundedness exactas y p < 0,01; sospechosas con precisión ≥ 0,65 y exhaustividad ≥ 0,85 | 4/4 |
| R5 | `canary_decision` | Decisión y nº de evaluaciones exactos en 3 flujos (sana, regresión grave, sin evidencia) | 3/3 |
| R6 | `build_regression_dashboard` | `lint` limpio y 8 consultas clave | 8/8 |

> 📏 Los umbrales salen de la solución de referencia con la semilla 42 fija en el notebook (R1, R3 y R5 se comparan con las funciones del paquete `supportops.regression` ejecutadas en vivo, así que no dependen de constantes).

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, dataclasses, json, math, re, time
from supportops.instrument import CompositeInstrumentor
from supportops.telemetry import OTelInstrumentor, setup_telemetry
from supportops.quality_eval import (CaseCollector, MockJudge, OnlineEvaluator, RateSampler, StratifiedSampler, VersionTagInstrumentor, confusion, simulate_feedback,
                                     wilson_interval)
from supportops.regression import (PROMPT_VERSIONS, Cusum, GoldenCase, LivePipeline, ReleaseGate, behavior_fingerprint, bootstrap_diff, canary_route, event_records,
                                   golden_to_tickets, js_divergence, ks_2samp, make_agent, mcnemar_exact, min_detectable_drop, novel_topic_tickets,
                                   peeking_false_positive_rate, psi, psi_categorical, replay, required_n_per_arm, run_version, select_golden, sprt_bernoulli,
                                   trace_shapes, trajectory_match, two_proportion_test, version_report)
STACK = check_stack([])
LIVE = False

from supportops.dashkit import Dashboard, lint, prom, loki, tempo, timeseries, stat, bargauge, traces_table, logs, table, heatmap, thresholds
from supportops.clock import stable_hex
from supportops.quality_eval import scrub_pii

## Paso 0 · Los datos (te lo damos hecho)

In [ ]:
J = MockJudge(SEED)
TK = generate_tickets(800, seed=SEED, fault_rates=DEFAULT_FAULT_RATES)
RES1, CASES1 = run_version("v1", TK)                                       # «producción» con v1
VERD1 = {c.ticket_id: J.evaluate(c) for c in CASES1}
FB1 = {f.ticket_id: f.rating for f in simulate_feedback(CASES1, seed=SEED)}
TOOLS1 = {r.ticket_id: r.tools_used for r in RES1}
SRC = {t.ticket_id: t for t in TK}

def run_telemetry(version, tickets, rate=0.3):
    ta = setup_telemetry("supportops-agent", mode="memory", set_global=False)
    te = setup_telemetry("supportops-evaluator", mode="memory", set_global=False)
    ev = OnlineEvaluator(J, RateSampler(rate, "tel"), te)
    agent, col = make_agent(version, instrumentors=[OTelInstrumentor(ta), VersionTagInstrumentor(version)], collector=CaseCollector(version, on_case=ev.submit))
    res = [agent.run(t) for t in tickets]; ev.run_pending(); ta.force_flush(); te.force_flush()
    return ta.span_records(), event_records(te), col.cases

sp1, ev1, _ = run_telemetry("v1", generate_tickets(500, seed=11, fault_rates=DEFAULT_FAULT_RATES))
sp2, ev2, cases_v2 = run_telemetry("v2", generate_tickets(500, seed=12, fault_rates=DEFAULT_FAULT_RATES))
SPANS, EVENTS = sp1 + sp2, ev1 + ev2                                       # LO ÚNICO que tienes para el Paso 4 (la verdad queda oculta en TRUTH_REG)
TRUTH_REG = {c.ticket_id for c in cases_v2 if "prompt_regression" in c.truth["faults"]}
print(f"producción v1: {len(CASES1)} casos, {sum(not v.passed for v in VERD1.values())} fallos del juez, {len(FB1)} valoraciones · telemetría A/B: {len(SPANS)} spans, {len(EVENTS)} eventos de evaluación")
print("un span invoke_agent:", {k: v for k, v in next(s for s in SPANS if s["name"].startswith("invoke_agent"))["attributes"].items() if k.startswith(("gen_ai.agent", "supportops.tool", "supportops.intent", "supportops.status"))})

## Paso 1 · El dataset de oro desde trazas

**TODO 1.** `build_golden(cases, verdicts, feedback, n_total, tools_by_ticket)` devuelve una lista de `dict` `{"case_id", "question", "reason", "source_ticket_id", "expected_tools", "weight"}` con estas reglas (las mismas que la lección, §3):

1. Reparte cada `EvalCase` en **un** cubo, por este orden de prioridad: `thumbs_down` (`feedback[ticket] < 0`), `judge_fail` (`not verdicts[ticket].passed`), `error` (`status` ∈ {error, max_steps, degraded}), `escalated`, `random_pass` (el resto con veredicto).
2. Recorre los cubos en ese orden; dentro de cada uno ordena por `stable_hex(8, "golden", 42, c.ticket_id)`. Cuotas: `thumbs_down`, `judge_fail` y `random_pass` hasta `n_total`; `error` y `escalated` hasta `max(n_total // 6, 4)`. Para al llegar a `n_total` casos.
3. **Deduplica** por `c.question.lower()` sin el sufijo « mi correo es…»: un duplicado **suma 1 al `weight`** del caso ya existente y no cuenta para la cuota. `weight` empieza en 1.
4. `question` = la pregunta **sin** el sufijo «Mi correo es…» y pasada por `scrub_pii`. `case_id` = `G-001`, `G-002`…
5. `expected_tools`: las herramientas usadas por el ticket (`tools_by_ticket`) **solo si** su veredicto aprobó y `status == "ok"`; si no, `[]` (la función del paquete, `select_golden`, rellena además esos huecos con la trayectoria más frecuente de la misma intención; aquí basta con `[]`).

<details>
<summary>🪜 Pista 1</summary>

Un `dict` de cubos `{razón: []}`; clasifica con un `if/elif` en el orden de prioridad. La clave de deduplicación es `re.sub(r'\s*mi correo es.*$', '', c.question.lower()).strip()`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Recorre `for reason, quota in cuotas:` ; `pool = sorted(buckets[reason], key=lambda c: stable_hex(8, 'golden', 42, c.ticket_id))`; para cada `c`: si la clave ya está en `seen`, `seen[key]['weight'] += 1` y `continue`; si no, comprueba `quota > 0` y `len(out) < n_total`.

</details>

In [ ]:
def build_golden(cases, verdicts, feedback, n_total, tools_by_ticket) -> list:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    G = build_golden(CASES1, VERD1, FB1, 120, TOOLS1)
    print(len(G), collections.Counter(g["reason"] for g in G), G[0])
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · La compuerta: comparación apareada

**TODO 2.** `compare_versions(base, cand, margin=0.03)` recibe dos listas de filas de `replay(...)` (dicts con `case_id`, `passed`, …) de **los mismos casos** y devuelve `{"b", "c", "diff", "p_value", "decision"}`:
`b` = casos que pasan con la base y fallan con la candidata, `c` = al revés, `diff` = calidad candidata − calidad base, `p_value` = `mcnemar_exact(b, c)` y `decision` = `"block"` si `diff < -margin` **y** `p_value < 0,05` (si no, `"ship"`). Lanza `ValueError` si las listas no tienen los mismos `case_id` en el mismo orden.

<details>
<summary>🪜 Pista 1</summary>

«Apareada» significa que cada caso se compara **consigo mismo** en las dos versiones. Los casos que pasan (o fallan) en ambas no informan sobre cuál es mejor; solo cuentan los **discordantes**: `b` = pasa con la base y falla con la candidata, `c` = al revés. McNemar exacto compara `b` y `c` (lo viste en la sección 4). Valida primero que los `case_id` coinciden y en el mismo orden (`ValueError` si no).

</details>

<details>
<summary>🪜 Pista 2</summary>

`b = sum(x['passed'] and not y['passed'] for x, y in zip(base, cand))` y `c` análogo; `diff` = media de `passed` de la candidata − media de la base; `p = mcnemar_exact(b, c)`. La decisión necesita **las dos condiciones a la vez**: `diff < -margin` (la caída es relevante) y `p < 0,05` (no es ruido).

</details>

In [ ]:
GOLD = select_golden(CASES1, VERD1, FB1, n_total=120, tools_by_ticket=TOOLS1)        # (lo da el paquete: úsalo en los pasos 2 y 3 aunque tu build_golden aún no esté)
GT = golden_to_tickets(GOLD, SRC)
REP = {v: replay(GOLD, GT, v, J) for v in ("v1", "v2", "v2.1")}

def compare_versions(base: list, cand: list, margin: float = 0.03) -> dict:
    # TODO 2
    raise NotImplementedError("TODO 2")

try:
    for v in ("v2", "v2.1"): print(v, compare_versions(REP["v1"], REP[v]))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · Vigilar el comportamiento sin juez

**TODO 3a.** `fingerprint(rows)` recibe filas por ticket (dicts con `n_llm_calls`, `n_tool_calls`, `input_tokens`, `output_tokens`, `cost_usd`, `latency_s`, `status`) y devuelve `{"llm_calls", "tool_calls", "no_tool_rate", "output_tokens", "input_tokens", "cost_usd", "latency_s", "error_rate"}`: las **medias** (y `no_tool_rate` = fracción con 0 herramientas, `error_rate` = fracción con `status` ∈ {error, max_steps}).

**TODO 3b.** `drift_flags(ref, cur)` recibe dos huellas y devuelve la **lista ordenada alfabéticamente** de métricas «marcadas»: `no_tool_rate` si sube más de **0,05** (absoluto); `error_rate` si sube más de **0,02**; y cualquier otra si cambia más de un **15 %** en relativo (`abs(cur-ref)/ref > 0,15`, ignorando `ref == 0`).

<details>
<summary>🪜 Pista 1</summary>

`fingerprint` es un resumen de una población: una **media por campo** (en ML, la «huella» de la distribución de salida de un modelo). `no_tool_rate` y `error_rate` son medias de booleanos (fracción de filas con `n_tool_calls == 0` o con `status` en `{error, max_steps}`). Un truco: `m = lambda k: sum(float(r[k]) for r in rows) / len(rows)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`drift_flags`: recorre las claves de la huella de referencia. Dos claves tienen regla **absoluta** (`no_tool_rate` > +0,05; `error_rate` > +0,02) y el resto una regla **relativa** (`abs(cur - ref) / ref > 0,15`, saltándote `ref == 0`). Fíjate en que los absolutos solo marcan **subidas**. Devuelve `sorted(...)`.

</details>

In [ ]:
def fingerprint(rows: list) -> dict:
    # TODO 3a
    raise NotImplementedError("TODO 3a")

def drift_flags(ref: dict, cur: dict) -> list:
    # TODO 3b
    raise NotImplementedError("TODO 3b")

ROWS = {v: [dict(r.to_dict(), n_tool_calls=r.n_tool_calls) for r in run_version(v, TK)[0]] for v in ("v1", "v2", "v2.1")}
try:
    FP = {v: fingerprint(ROWS[v]) for v in ROWS}
    print({v: {k: round(x, 3) for k, x in f.items()} for v, f in FP.items()})
    print({v: drift_flags(FP["v1"], FP[v]) for v in ("v2", "v2.1")})
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Cazar la regresión solo con telemetría

**TODO 4.** `find_regression(spans, events)` recibe la lista de spans (`SPAN_SCHEMA`) y los eventos de evaluación (`{"attributes": {...}}`) y devuelve `{"culprit", "groundedness", "p_value", "suspects"}`:

- `groundedness`: `dict` versión → fracción de eventos `gen_ai.evaluation.name == "groundedness"` con `score.label == "pass"` (agrupa por `gen_ai.agent.version` del evento).
- `culprit`: la versión con **menor** groundedness si la diferencia con la otra es significativa (`two_proportion_test(...)["p_value"] < 0,01`), si no `None`.
- `suspects`: `trace_id` de los spans `invoke_agent` de la versión culpable con `supportops.tool_calls == 0`, `supportops.status == "ok"` e `supportops.intent` ∈ {`order_status`, `refund`, `kb_question`}.

<details>
<summary>🪜 Pista 1</summary>

Los spans `invoke_agent` tienen `attributes['gen_ai.operation.name'] == 'invoke_agent'`, `gen_ai.agent.version`, `supportops.tool_calls`, `supportops.status` y `supportops.intent`. Cada evento lleva `gen_ai.agent.version`, `gen_ai.evaluation.name` y `gen_ai.evaluation.score.label`.

</details>

In [ ]:
def find_regression(spans: list, events: list) -> dict:
    # TODO 4
    raise NotImplementedError("TODO 4")

try:
    FR = find_regression(SPANS, EVENTS)
    print(FR["culprit"], {k: round(v, 3) for k, v in FR["groundedness"].items()}, f"p={FR['p_value']:.2g}", len(FR["suspects"]), "trazas sospechosas")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · Canario con decisión secuencial

**TODO 5.** `canary_decision(fails, p0, p1, alpha=0.05, beta=0.2)` recibe una lista de booleanos (`True` = fallo de calidad en cada evaluación del canario, en orden) y devuelve `(decisión, n)`: tras cada observación `i` (1-based) calcula
\( \Lambda_i = k\ln\frac{p_1}{p_0} + (i-k)\ln\frac{1-p_1}{1-p_0} \) y devuelve `("rollback", i)` si \( \Lambda_i \ge \ln\frac{1-\beta}{\alpha} \), `("promote", i)` si \( \Lambda_i \le \ln\frac{\beta}{1-\alpha} \); si termina sin cruzar ninguna cota, `("continue", len(fails))`.

<details>
<summary>🪜 Pista 1</summary>

Calcula las dos cotas **una vez**, fuera del bucle: `upper = ln((1−β)/α)` y `lower = ln(β/(1−α))`. Dentro del bucle lleva la cuenta de fallos `k` y, con `i` observaciones, evalúa \( \Lambda_i \). Es una prueba de razón de verosimilitudes acumulada: cada fallo la **sube** (porque `p1 > p0`) y cada éxito la **baja**.

</details>

<details>
<summary>🪜 Pista 2</summary>

Comprueba primero `rollback` (Λ ≥ cota superior) y luego `promote` (Λ ≤ cota inferior); devuelve `(decisión, i)` en cuanto cruces una. Si terminas el bucle sin cruzar, `("continue", len(fails))`: «todavía no hay datos suficientes», que es una respuesta legítima y no un error.

</details>

In [ ]:
def canary_decision(fails: list, p0: float, p1: float, alpha: float = 0.05, beta: float = 0.2) -> tuple:
    # TODO 5
    raise NotImplementedError("TODO 5")

FAILS = {v: [not x.scores["groundedness"].label == "pass" for x in [J.evaluate(c) for c in run_version(v, TK[:400])[1]]] for v in ("v1", "v2")}
try:
    print({v: canary_decision(f, 0.07, 0.15) for v, f in FAILS.items()}, canary_decision(FAILS["v1"][:15], 0.07, 0.15))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · El panel de regresión como código

**TODO 6.** `build_regression_dashboard()` devuelve el `dict` de un dashboard con **uid** `regresion-versiones`, título «15 · Regresión por versión (proyecto)» y **al menos 8 paneles con `description`** que incluyan estas consultas:

1. **Calidad por versión**: `sum by (gen_ai_agent_version)` sobre `supportops_eval_results_total` con `gen_ai_evaluation_score_label="pass"`.
2. **Tráfico por versión**: `supportops_eval_population_total` agrupado por `gen_ai_agent_version`.
3. **Herramientas por ticket**: `gen_ai_invoke_agent_tool_calls_sum` / `gen_ai_invoke_agent_tool_calls_count`.
4. **Latencia p95 clásica**: `histogram_quantile(0.95, …gen_ai_invoke_agent_duration_seconds_bucket…)`.
5. **Coste por ticket**: `supportops_agent_request_cost_USD_sum` / `_count`.
6. **Errores**: `supportops_agent_requests_total` con `supportops_status=~"error|max_steps"`.
7. **Trazas sospechosas (Tempo)**: contiene `span.gen_ai.agent.version` y `span.supportops.tool_calls = 0`.
8. **Evaluaciones fallidas por versión (Loki)**: contiene `gen_ai_evaluation_score_label="fail"` y `gen_ai_agent_version`.

<details>
<summary>🪜 Pista 1</summary>

Es el mismo ejercicio del módulo 10, con otras consultas: `Dashboard(uid, titulo, tags=[...], description=...)` y `d.add(timeseries(titulo, [prom(expr, leyenda)], unit=..., w=..., description=...), ...)`. Las consultas de Prometheus están en la lista del enunciado; para las de Tempo y Loki usa `traces_table(...)` y `logs(...)`. Termina con `return d.to_dict()` y comprueba con `lint(D)` (la celda de prueba ya lo hace).

</details>

<details>
<summary>🪜 Pista 2</summary>

Para los cocientes («herramientas por ticket», «coste por ticket») usa `sum(rate(X_sum[5m])) / sum(rate(X_count[5m]))`; para la calidad, `pass` dividido entre el total de evaluaciones, ambos `sum by (gen_ai_agent_version)`. Exige 8 paneles **con `description`**: escribe una frase que diga qué mide y cómo se lee (qué sube/baja cuando hay regresión).

</details>

In [ ]:
def build_regression_dashboard() -> dict:
    # TODO 6
    raise NotImplementedError("TODO 6")

try:
    D = build_regression_dashboard(); print(len(D["panels"]), "paneles · lint:", lint(D) or "OK")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 7 · Rúbrica

In [ ]:
def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": det})
    # R1
    try:
        G = build_golden(CASES1, VERD1, FB1, 120, TOOLS1); ref = select_golden(CASES1, VERD1, FB1, n_total=120, tools_by_ticket=TOOLS1)
        sig = lambda x: [(g["source_ticket_id"], g["reason"], g["weight"]) for g in x] if isinstance(x[0], dict) else [(g.source_ticket_id, g.reason, g.weight) for g in x]
        keys = [re.sub(r"\s*mi correo es.*$", "", g["question"].lower()).strip() for g in G]
        checks = {"mismos casos, razones y pesos": sig(G) == sig(ref), "sin duplicados": len(set(keys)) == len(keys), "sin PII": all("@" not in g["question"] and "correo es" not in g["question"].lower() for g in G),
                  "ids y pesos": [g["case_id"] for g in G] == [f"G-{i + 1:03d}" for i in range(len(G))] and all(g["weight"] >= 1 for g in G), "trayectorias": all(g["expected_tools"] == (r.expected_tools if (VERD1[g["source_ticket_id"]].passed and next(c for c in CASES1 if c.ticket_id == g["source_ticket_id"]).status == "ok") else []) for g, r in zip(G, ref))}
        add("R1", "build_golden: 5 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or f"todas OK · {len(G)} casos"))
    except Exception as e: add("R1", "build_golden", False, f"{type(e).__name__}: {e}")
    # R2
    try:
        a, b = compare_versions(REP["v1"], REP["v2"]), compare_versions(REP["v1"], REP["v1"])
        bb = sum(x["passed"] and not y["passed"] for x, y in zip(REP["v1"], REP["v2"])); cc = sum((not x["passed"]) and y["passed"] for x, y in zip(REP["v1"], REP["v2"]))
        def raises(): 
            try: compare_versions(REP["v1"], REP["v2"][:-1])
            except ValueError: return True
            except Exception: return False
            return False
        checks = {"v2 bloqueada": a["decision"] == "block", "v1 vs v1 pasa": b["decision"] == "ship" and b["p_value"] == 1.0, "b, c y p exactos": (a["b"], a["c"]) == (bb, cc) and abs(a["p_value"] - mcnemar_exact(bb, cc)) < 1e-12, "ValueError si no son los mismos casos": raises()}
        add("R2", "compare_versions: 4 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or f"todas OK · v1→v2: {a}"))
    except Exception as e: add("R2", "compare_versions", False, f"{type(e).__name__}: {e}")
    # R3
    try:
        FPs = {v: fingerprint(ROWS[v]) for v in ROWS}
        ref = {v: behavior_fingerprint(ROWS[v]) for v in ROWS}
        same = all(abs(FPs[v][k] - ref[v][k]) < 1e-9 for v in ROWS for k in ("llm_calls", "tool_calls", "no_tool_rate", "output_tokens", "input_tokens", "cost_usd", "latency_s", "error_rate"))
        f2, f21 = drift_flags(FPs["v1"], FPs["v2"]), drift_flags(FPs["v1"], FPs["v2.1"])
        checks = {"huella exacta": same, "v2 marca no_tool_rate y tool_calls": "no_tool_rate" in f2 and "tool_calls" in f2, "v2.1 no marca no_tool_rate (sube < 0,05)": "no_tool_rate" not in f21, "orden alfabético": f2 == sorted(f2)}
        add("R3", "fingerprint y drift_flags: 4 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or f"v2: {f2} · v2.1: {f21}"))
    except Exception as e: add("R3", "fingerprint/drift_flags", False, f"{type(e).__name__}: {e}")
    # R4
    try:
        FR = find_regression(SPANS, EVENTS); sus = set(FR["suspects"])
        tid2ticket = {s["trace_id"]: s["attributes"].get("supportops.ticket.id") for s in SPANS if s["name"].startswith("invoke_agent")}
        got = {tid2ticket[t] for t in sus}
        prec = len(got & TRUTH_REG) / max(len(got), 1); rec = len(got & TRUTH_REG) / len(TRUTH_REG)
        ok_ev = {}
        for e in EVENTS:
            a_ = e["attributes"]
            if a_.get("gen_ai.evaluation.name") == "groundedness": ok_ev.setdefault(a_["gen_ai.agent.version"], []).append(a_["gen_ai.evaluation.score.label"] == "pass")
        ref_g = {v: sum(x) / len(x) for v, x in ok_ev.items()}
        checks = {"culpable = v2": FR["culprit"] == "v2", "groundedness exacta": all(abs(FR["groundedness"][v] - ref_g[v]) < 1e-9 for v in ref_g), "p < 0,01": FR["p_value"] < 0.01, "precisión ≥ 0,65 y exhaustividad ≥ 0,85": prec >= 0.65 and rec >= 0.85}
        add("R4", "find_regression: 4 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK") + f" · precisión={prec:.2f} exhaustividad={rec:.2f}")
    except Exception as e: add("R4", "find_regression", False, f"{type(e).__name__}: {e}")
    # R5
    try:
        outs = [canary_decision(FAILS["v1"], 0.07, 0.15), canary_decision(FAILS["v2"], 0.07, 0.15), canary_decision(FAILS["v1"][:15], 0.07, 0.15)]
        def ref_dec(f, p0, p1):
            k = 0
            for i, x in enumerate(f, 1):
                k += x; d, _ = sprt_bernoulli(k, i, p0, p1)
                if d != "continue": return (d, i)
            return ("continue", len(f))
        refs = [ref_dec(FAILS["v1"], 0.07, 0.15), ref_dec(FAILS["v2"], 0.07, 0.15), ref_dec(FAILS["v1"][:15], 0.07, 0.15)]
        add("R5", "canary_decision: 3 flujos", outs == refs and outs[1][0] == "rollback" and outs[2][0] == "continue", f"{outs} (ref {refs})")
    except Exception as e: add("R5", "canary_decision", False, f"{type(e).__name__}: {e}")
    # R6
    try:
        D = build_regression_dashboard()
        exprs = []
        for p in D["panels"]:
            for t in p.get("targets", []): exprs.append(t.get("expr") or t.get("query") or "")
        has = lambda *pats: any(all(re.search(p, e) for p in pats) for e in exprs)
        checks = {"lint limpio, uid y ≥ 8 paneles": not lint(D) and D["uid"] == "regresion-versiones" and len(D["panels"]) >= 8,
                  "calidad por versión": has(r"sum by \(gen_ai_agent_version\)", r"supportops_eval_results_total", r'score_label="pass"'), "tráfico por versión": has(r"supportops_eval_population_total", r"gen_ai_agent_version"),
                  "herramientas por ticket": has(r"gen_ai_invoke_agent_tool_calls_sum", r"gen_ai_invoke_agent_tool_calls_count"), "latencia p95": has(r"histogram_quantile\(0\.95", r"gen_ai_invoke_agent_duration_seconds_bucket"),
                  "coste por ticket": has(r"supportops_agent_request_cost_USD_sum", r"supportops_agent_request_cost_USD_count"), "errores": has(r"supportops_agent_requests_total", r"error\|max_steps"),
                  "Tempo sospechosas": has(r"span\.gen_ai\.agent\.version", r"span\.supportops\.tool_calls = 0"), "Loki fallidas": has(r'gen_ai_evaluation_score_label="fail"', r"gen_ai_agent_version")}
        add("R6", "build_regression_dashboard: lint + 8 consultas", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R6", "build_regression_dashboard", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Paso final (opcional) · Míralo funcionar

Con el stack: guarda tu dashboard (`Dashboard.save(...)` en `stack/grafana/dashboards/`) y ejecuta un despliegue en canario en vivo con `LivePipeline(("v1", "v2"), mode="otlp", sampler=RateSampler(0.6))`
(ver la sección 9 de la lección). Espera ≈ 20 s: «Calidad por versión» debe separar v1 y v2 mientras latencia y coste de v2 parecen mejores.

## Retos extra (nivel experto)

1. **Detector sin versión**: si los spans NO llevaran `gen_ai.agent.version`, ¿podrías inferir el despliegue? Detecta el punto de cambio con CUSUM sobre `tool_calls` por ticket en una serie temporal y compáralo con el momento del despliegue.
2. **Potencia del dataset**: calcula con simulación (bootstrap sobre `REP`) la probabilidad de bloquear v2.1 con 60, 120 y 240 casos de oro. ¿Dónde se cruza el 80 %?
3. **Estratificación del dataset**: reponde los casos por `weight` y por intención para que la métrica del replay refleje la producción; ¿cambia la decisión sobre v2.1?
4. **Canario multi-criterio**: extiende `canary_decision` para vigilar a la vez calidad, `no_tool_rate` y latencia (corrección de Bonferroni o *alpha spending*).
5. **Reversión automática**: conecta `canary_decision` con `LivePipeline` y revierte el enrutamiento cuando decida `rollback`; mide cuántos clientes vieron v2 antes de la reversión.

## Reflexión (producción e hilo conductor)

- Tu detector estructural tiene falsos positivos (peticiones legítimas sin herramienta). ¿Cómo evitarías que una alerta basada en él despierte a alguien por una campaña que pregunta cosas nuevas? (módulo 16).
- ¿Qué metadatos de la traza necesitas conservar para reconstruir el dataset de oro dentro de 3 meses? ¿Y cuáles deberías **borrar** por privacidad (módulo 20)?
- Con v2.1 la compuerta bloquea por poco: ¿qué decisión de negocio tomas (coste de una regresión residual del 5 % frente al de retrasar el release)?
- En el capstone el agente es multi-agente (módulo 11): ¿qué cambia en la huella de comportamiento cuando hay handoffs?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def build_golden(cases, verdicts, feedback, n_total, tools_by_ticket):
    buckets = {"thumbs_down": [], "judge_fail": [], "error": [], "escalated": [], "random_pass": []}
    for c in cases:
        v = verdicts.get(c.ticket_id)
        if feedback.get(c.ticket_id, 0) < 0: buckets["thumbs_down"].append(c)
        elif v is not None and not v.passed: buckets["judge_fail"].append(c)
        elif c.status in ("error", "max_steps", "degraded"): buckets["error"].append(c)
        elif c.status == "escalated": buckets["escalated"].append(c)
        elif v is not None: buckets["random_pass"].append(c)
    quota = {"thumbs_down": n_total, "judge_fail": n_total, "error": max(n_total // 6, 4), "escalated": max(n_total // 6, 4), "random_pass": n_total}
    out, seen = [], {}
    for reason in ("thumbs_down", "judge_fail", "error", "escalated", "random_pass"):
        for c in sorted(buckets[reason], key=lambda c: stable_hex(8, "golden", 42, c.ticket_id)):
            if len(out) >= n_total or quota[reason] <= 0: break
            key = re.sub(r"\s*mi correo es.*$", "", c.question.lower()).strip()
            if key in seen:
                seen[key]["weight"] += 1; continue
            quota[reason] -= 1
            v = verdicts.get(c.ticket_id)
            tools = list(tools_by_ticket.get(c.ticket_id, [])) if (v is not None and v.passed and c.status == "ok") else []
            g = {"case_id": f"G-{len(out) + 1:03d}", "question": re.sub(r"\s*Mi correo es.*$", "", scrub_pii(c.question)), "reason": reason, "source_ticket_id": c.ticket_id, "expected_tools": tools, "weight": 1.0}
            seen[key] = g; out.append(g)
    return out

def compare_versions(base, cand, margin=0.03):
    if [r["case_id"] for r in base] != [r["case_id"] for r in cand]: raise ValueError("las listas deben ser de los mismos casos")
    b = sum(bool(x["passed"]) and not bool(y["passed"]) for x, y in zip(base, cand)); c = sum((not bool(x["passed"])) and bool(y["passed"]) for x, y in zip(base, cand))
    diff = np.mean([bool(y["passed"]) for y in cand]) - np.mean([bool(x["passed"]) for x in base]); p = mcnemar_exact(b, c)
    return {"b": b, "c": c, "diff": float(diff), "p_value": p, "decision": "block" if diff < -margin and p < 0.05 else "ship"}

def fingerprint(rows):
    n = len(rows); m = lambda k: sum(float(r[k]) for r in rows) / n
    return {"llm_calls": m("n_llm_calls"), "tool_calls": m("n_tool_calls"), "no_tool_rate": sum(int(r["n_tool_calls"]) == 0 for r in rows) / n, "output_tokens": m("output_tokens"),
            "input_tokens": m("input_tokens"), "cost_usd": m("cost_usd"), "latency_s": m("latency_s"), "error_rate": sum(r["status"] in ("error", "max_steps") for r in rows) / n}

def drift_flags(ref, cur):
    flags = []
    for k in ref:
        if k == "no_tool_rate": hit = cur[k] - ref[k] > 0.05
        elif k == "error_rate": hit = cur[k] - ref[k] > 0.02
        else: hit = ref[k] != 0 and abs(cur[k] - ref[k]) / ref[k] > 0.15
        if hit: flags.append(k)
    return sorted(flags)

def find_regression(spans, events):
    cnt = {}
    for e in events:
        a = e["attributes"]
        if a.get("gen_ai.evaluation.name") == "groundedness":
            k = cnt.setdefault(a["gen_ai.agent.version"], [0, 0]); k[1] += 1; k[0] += a["gen_ai.evaluation.score.label"] == "pass"
    g = {v: k / n for v, (k, n) in cnt.items()}
    (va, (ka, na)), (vb, (kb, nb)) = sorted(cnt.items())[:2]
    p = two_proportion_test(ka, na, kb, nb)["p_value"]
    worst = min(g, key=g.get)
    culprit = worst if p < 0.01 else None
    sus = [s["trace_id"] for s in spans if s["attributes"].get("gen_ai.operation.name") == "invoke_agent" and s["attributes"].get("gen_ai.agent.version") == culprit
           and s["attributes"].get("supportops.tool_calls") == 0 and s["attributes"].get("supportops.status") == "ok" and s["attributes"].get("supportops.intent") in ("order_status", "refund", "kb_question")]
    return {"culprit": culprit, "groundedness": g, "p_value": p, "suspects": sus}

def canary_decision(fails, p0, p1, alpha=0.05, beta=0.2):
    up, lo = math.log((1 - beta) / alpha), math.log(beta / (1 - alpha)); k = 0
    for i, f in enumerate(fails, 1):
        k += bool(f); llr = k * math.log(p1 / p0) + (i - k) * math.log((1 - p1) / (1 - p0))
        if llr >= up: return ("rollback", i)
        if llr <= lo: return ("promote", i)
    return ("continue", len(fails))

def build_regression_dashboard():
    d = Dashboard("regresion-versiones", "15 · Regresión por versión (proyecto)", tags=["curso", "regresion"], description="Calidad, comportamiento y señales clásicas por versión del agente.")
    d.add(
        timeseries("Calidad por versión", [prom('sum by (gen_ai_agent_version) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness",gen_ai_evaluation_score_label="pass"}[5m])) / sum by (gen_ai_agent_version) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness"}[5m]))', "{{gen_ai_agent_version}}")], unit="percentunit", w=12, description="Groundedness por versión."),
        timeseries("Tráfico por versión", [prom("sum by (gen_ai_agent_version) (rate(supportops_eval_population_total[5m]))", "{{gen_ai_agent_version}}")], w=12, description="Peticiones por segundo por versión (reparto del canario)."),
        timeseries("Herramientas por ticket", [prom("sum(rate(gen_ai_invoke_agent_tool_calls_sum[5m])) / sum(rate(gen_ai_invoke_agent_tool_calls_count[5m]))", "herramientas/ticket")], w=8, description="Baja cuando el agente deja de consultar."),
        timeseries("Latencia p95 (clásica)", [prom("histogram_quantile(0.95, sum by (le) (rate(gen_ai_invoke_agent_duration_seconds_bucket[5m])))", "p95")], unit="s", w=8, description="La señal clásica mejora con la regresión."),
        timeseries("Coste por ticket", [prom("sum(rate(supportops_agent_request_cost_USD_sum[5m])) / sum(rate(supportops_agent_request_cost_USD_count[5m]))", "USD/ticket")], unit="currencyUSD", w=8, description="Coste medio por ticket (ilustrativo)."),
        timeseries("Errores", [prom('sum(rate(supportops_agent_requests_total{supportops_status=~"error|max_steps"}[5m])) / sum(rate(supportops_agent_requests_total[5m]))', "error ratio")], unit="percentunit", w=8, description="Tasa de error: no cambia con la regresión."),
        traces_table("Trazas sospechosas (Tempo)", '{ name =~ "invoke_agent.*" && span.gen_ai.agent.version = "v2" && span.supportops.tool_calls = 0 && span.supportops.intent =~ "order_status|refund" }', w=8, description="Tickets de v2 que debían usar una herramienta y no la usaron."),
        logs("Evaluaciones fallidas por versión (Loki)", '{service_name="supportops-evaluator"} | gen_ai_evaluation_score_label="fail" | gen_ai_agent_version="v2"', w=8, description="Eventos de evaluación fallidos de la versión v2."),
    )
    return d.to_dict()

display(run_rubric())